### Installation

In [1]:
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !uv pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [2]:
#@title Colab Extra Install { display-mode: "form" }
import os
# !pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !uv pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

Using Python 3.12.13 environment at: /usr
Resolved 18 packages in 32ms
Uninstalled 2 packages in 210ms
Installed 2 packages in 189ms
 - huggingface-hub==1.20.1
 + huggingface-hub==0.36.2
 - transformers==5.5.0
 + transformers==4.56.2
Using Python 3.12.13 environment at: /usr
Resolved 1 package in 2ms
Uninstalled 1 package in 2ms
Installed 1 package in 17ms
 - trl==0.24.0
 + trl==0.22.2


### Unsloth

Load up `Llama 3.1 8B Instruct`, and set parameters

In [3]:
from unsloth import FastLanguageModel
import torch

# Set the maximum sequence length (number of tokens) for the model to handle.
# Increasing this allows the model to process longer input and output sequences,
# which can be helpful for problems requiring extended reasoning or context.
max_seq_length = 1024 # Can increase for longer reasoning traces

# Set the rank for LoRA (Low-Rank Adaptation). Increasing the rank makes the
# adaptation more expressive ("smarter") but also slower and more memory intensive.
lora_rank = 32 # Larger rank = smarter, but slower

# Load the pretrained Llama 3.1 8B Instruct model using Unsloth's FastLanguageModel.
# - model_name: specifies the weights to use.
# - max_seq_length: the sequence length that the model will accept.
# - load_in_4bit: loads the model in quantized 4-bit precision for memory efficiency.
# - fast_inference: enables fast inference mode (with vLLM backend).
# - max_lora_rank: maximum allowable LoRA rank.
# - gpu_memory_utilization: percent of GPU VRAM to use (lower if OOM errors).
model, tokenizer = FastLanguageModel.from_pretrained(
    # model_name = "unsloth/meta-Llama-3.1-8B-Instruct",
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,        # False will use default (usually 16bit, more precision)
    fast_inference = True,      # Enable vllm fast inference for speed
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Lower this value if you run out of memory.
)

# Prepare the model for Parameter-Efficient Fine-Tuning (PEFT) using LoRA.
# - r: LoRA rank (controls adaptation capacity).
# - target_modules: list of module names to apply LoRA adapters to.
#   (QKVO are projection layers; gate/up/down_proj are MLP/attention components).
#   Remove modules here if you get out-of-memory errors.
# - lora_alpha: scale factor for LoRA.
# - use_gradient_checkpointing: enables gradient checkpointing to reduce memory usage,
#   especially helpful for long context training/fine-tuning.
# - random_state: fixes the random seed for reproducibility.
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested values include 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",      # Attention projection layers
        "gate_proj", "up_proj", "down_proj",         # Feedforward MLP projection layers
    ], # Remove QKVO if out of memory; fewer adapters reduce memory but limit learning capacity
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable efficient long context fine-tuning
    random_state = 3407, # Set a seed for reproducibility
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 06-23 08:01:43 [__init__.py:244] Automatically detected platform cuda.
ERROR 06-23 08:01:46 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 06-23 08:01:55 [vllm_utils.py:739] Unsloth: Patching vLLM v1 graph capture
INFO 06-23 08:01:55 [vllm_utils.py:768] Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Standb

`torch_dtype` is deprecated! Use `dtype` instead!


WARNING 06-23 08:02:19 [config.py:3371] Casting torch.bfloat16 to torch.float16.
INFO 06-23 08:02:19 [config.py:1472] Using max model len 1024
WARNING 06-23 08:02:19 [arg_utils.py:1735] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 06-23 08:02:19 [config.py:2285] Chunked prefill is enabled with max_num_batched_tokens=4096.
Unsloth: vLLM Bitsandbytes config using kwargs = {'load_in_8bit': False, 'load_in_4bit': True, 'bnb_4bit_compute_dtype': 'float16', 'bnb_4bit_quant_storage': 'uint8', 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': ['lm_head', 'multi_modal_projector', 'merger', 'modality_projection', 'model.layers.1.mlp'], 'llm_int8_threshold': 6.0}
INFO 06-23 08:02:19 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.2) with config: model='unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit', speculative_config=None, tokenize

model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

INFO 06-23 08:02:51 [weight_utils.py:308] Time spent downloading weights for unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit: 28.476276 seconds
INFO 06-23 08:02:51 [weight_utils.py:345] No model.safetensors.index.json found in remote.


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth 2026.6.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


### Data Prep
<a name="Data"></a>

We directly leverage [@willccbb](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb) for data prep and all reward functions. You are free to create your own!

In [4]:
import re
from datasets import load_dataset, Dataset

# Define a consistent system prompt that instructs models to respond
# in a two-part XML format: <reasoning> ... </reasoning><answer> ... </answer>
SYSTEM_PROMPT = """
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

# Example XML Chain-of-Thought format string for use in data formatting/generation.
XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

def extract_xml_answer(text: str) -> str:
    """
    Extracts the answer section wrapped in <answer>...</answer> tags from a string.
    Returns the inner text of the answer tag, with whitespace stripped.
    """
    answer = text.split("<answer>")[-1]          # Get text after the opening <answer>
    answer = answer.split("</answer>")[0]        # Get text before the closing </answer>
    return answer.strip()

def extract_hash_answer(text: str) -> str | None:
    """
    Extracts the answer after '####' typically included as the answer delimiter
    in the GSM8K dataset. Returns None if '####' is not found.
    """
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

# Uncomment the middle messages if you want to try 1-shot prompting.
def get_gsm8k_questions(split = "train") -> Dataset:
    """
    Loads the GSM8K dataset and prepares each sample by:
        - Creating a prompt consisting of a system and user message.
        - Extracting the answer string from the 'answer' field using extract_hash_answer().
    Returns a HuggingFace Dataset object.
    """
    data = load_dataset('openai/gsm8k', 'main')[split]  # Load the specified split
    data = data.map(lambda x: {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},         # Add the consistent system prompt
            {'role': 'user', 'content': x['question']}            # Add the user message with the GSM8K question
        ],
        'answer': extract_hash_answer(x['answer'])                # Parse and extract the answer string from solution
    })
    return data

# Load the standard dataset (train split by default)
dataset = get_gsm8k_questions()

#################
# Reward Functions
#################

def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """
    For each completion, extract its content, compare the extracted <answer> section
    to the ground-truth answer, and reward 2.0 for an exact match (else 0.0).
    Prints details of the comparison for debugging.
    """
    responses = [completion[0]['content'] for completion in completions]
    q = prompts[0][-1]['content']
    extracted_responses = [extract_xml_answer(r) for r in responses]
    print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}")
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

def int_reward_func(completions, **kwargs) -> list[float]:
    """
    For each completion, extracts the <answer> string and returns 0.5
    if the answer is a digit (for checking if the output is an integer).
    """
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """
    Checks if each completion exactly conforms to the strict XML reasoning/answer format,
    requiring newlines and correct tag placement. Returns 0.5 if matched, else 0.0.
    """
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """
    Looser check for the XML reasoning/answer format, not requiring strict newlines or placement.
    Returns 0.5 if the overall pattern is present, else 0.0.
    """
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def count_xml(text) -> float:
    """
    Heuristic function: Examines the serialized text and gives a partial reward based on
    the number and placement of XML tags/newlines for <reasoning> and <answer>.
    Penalizes extra trailing text after the closing </answer> tag.
    """
    count = 0.0
    # Each correctly placed tag gives partial credit
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        # Penalize any extra output after the close
        count -= len(text.split("\n</answer>\n")[-1]) * 0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1) * 0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    """
    For each completion, returns a float reward based on the presence and
    positioning of XML tags as checked by count_xml().
    """
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [12]:
# Define the maximum length of the input prompt (how much of the question or instruction to feed in)
max_prompt_length = 256

# Import the GRPOConfig class for configuration and the GRPOTrainer for training from the TRL (TRL = Transformer Reinforcement Learning) library
from trl import GRPOConfig, GRPOTrainer

# Set up the training configuration for our GRPOTrainer.
# Each parameter is explained below:
training_args = GRPOConfig(
    learning_rate = 5e-6,               # The initial learning rate for the Adam optimizer
    adam_beta1 = 0.9,                   # The beta1 hyperparameter for Adam (momentum decay)
    adam_beta2 = 0.99,                  # The beta2 hyperparameter for Adam (RMSprop smoothing)
    weight_decay = 0.001,               # L2 weight decay to regularize model parameters
    warmup_ratio = 0.1,                 # Fraction of steps spent linearly increasing LR at the start
    lr_scheduler_type = "cosine",       # Use a cosine annealing schedule for learning rate
    optim = "paged_adamw_8bit",         # Optimizer type, here leveraging paged AdamW with 8bit precision for efficiency
    logging_steps = 1,                  # Log training progress every step
    per_device_train_batch_size = 1,    # How many samples per device (GPU/CPU) per training step
    gradient_accumulation_steps = 1,    # Accumulate gradients over this many steps before updating weights. Increase to smooth training and use less memory.
    num_generations = 6,                # How many completions to generate per training instance (decrease if you run out of memory)
    max_prompt_length = max_prompt_length,                         # Maximum length for the input prompt
    max_completion_length = max_seq_length - max_prompt_length,    # Maximum length for generated completion (output tokens)
    # num_train_epochs = 1,             # Optionally set this for full training; here we use max_steps instead
    max_steps = 250,                    # Total number of steps to train for (early stop after this many steps)
    save_steps = 250,                   # Frequency of saving model checkpoints (here, only at the end)
    max_grad_norm = 0.1,                # Maximum norm for gradient clipping (prevents exploding gradients)
    report_to = "none",                 # Reporting service; "none" disables logging to external tools (set to "wandb" to enable Weights & Biases)
    output_dir = "outputs",             # Directory where output files and checkpoints are saved
)

# This configuration is passed to the GRPOTrainer, which will use it to fine-tune your model using RLHF.

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 6


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |

In [6]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        xmlcount_reward_func,
        soft_format_reward_func,
        strict_format_reward_func,
        int_reward_func,
        correctness_reward_func,
    ],
    args = training_args,
    train_dataset = dataset,
)
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,473 | Num Epochs = 1 | Total steps = 250
O^O/ \_/ \    Batch size per device = 6 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (6 x 1 x 1) = 6
 "-____-"     Trainable parameters = 48,627,712 of 3,261,377,536 (1.49% trained)


Unsloth: Will smartly offload gradients to save VRAM!
-------------------- Question:
A concert ticket costs $40. Mr. Benson bought 12 tickets and received a 5% discount for every ticket bought that exceeds 10. How much did Mr. Benson pay in all? 
Answer:
476 
Response:
<reasoning>
To find out the total amount paid, we need to first calculate the price of the prices 11 or more tickets and then apply a 5% discount from the higher end of those group of tickets
First we need to find out how many tickets Mr. Benson bought that qualifies for the 5% discount > 10
He bought 12 tickets and so 11 or more so we'll need to cut off a couple.
Mr. Benson  bought 11 tickets that qualify for the discount
Now we need to use 10 as a decimal and 11 instead of an integer
10 as a decimal is.10
11 x $.40 =
$4.40
  But then we apply that 5% to the.10 = (0.05)(10) = 0.5
do not forget this 0.5 is a discount i.e discount 5 cents more
So now,

 Customizing $4.4 with $.05 = $4.45
Now that we know how to calculate 

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / xmlcount_reward_func / mean,rewards / xmlcount_reward_func / std,rewards / soft_format_reward_func / mean,rewards / soft_format_reward_func / std,rewards / strict_format_reward_func / mean,rewards / strict_format_reward_func / std,rewards / int_reward_func / mean,rewards / int_reward_func / std,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std
1,0.157500,-0.294667,0.296143,215.166672,171.000000,303.000000,0.000000,215.166672,171.000000,303.000000,0.000000,-0.294667,0.296143,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.118600,0.230667,1.001558,355.666687,6.000000,768.000000,0.166667,273.200012,6.000000,517.000000,0.000000,-0.186000,0.448834,0.000000,0.000000,0.000000,0.000000,0.083333,0.204124,0.333333,0.816497
3,0.081100,-0.224000,0.390794,214.833344,169.000000,267.000000,0.000000,214.833344,169.000000,267.000000,0.000009,-0.224000,0.390794,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.103800,-0.164500,0.316169,232.333344,132.000000,278.000000,0.000000,232.333344,132.000000,278.000000,0.000006,-0.247833,0.282719,0.000000,0.000000,0.000000,0.000000,0.083333,0.204124,0.000000,0.000000
5,-0.005800,-0.086167,0.138027,138.833344,106.000000,183.000000,0.000000,138.833344,106.000000,183.000000,0.000007,-0.086167,0.138027,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,-0.093200,0.222000,0.706620,244.333344,193.000000,301.000000,0.000000,244.333344,193.000000,301.000000,0.000007,-0.194667,0.347820,0.000000,0.000000,0.000000,0.000000,0.083333,0.204124,0.333333,0.816497
7,0.190600,-0.106500,0.267196,197.333344,129.000000,285.000000,0.000000,197.333344,129.000000,285.000000,0.000006,-0.106500,0.267196,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,0.364100,-0.396500,0.434644,206.000000,73.000000,330.000000,0.000000,206.000000,73.000000,330.000000,0.000011,-0.396500,0.434644,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9,0.040400,0.412333,0.896679,191.833344,131.000000,353.000000,0.000000,191.833344,131.000000,353.000000,0.000010,-0.087667,0.122609,0.000000,0.000000,0.000000,0.000000,0.166667,0.258199,0.333333,0.816497
10,0.107800,-0.115000,0.260990,198.333344,140.000000,245.000000,0.000000,198.333344,140.000000,245.000000,0.000008,-0.198333,0.237871,0.000000,0.000000,0.000000,0.000000,0.083333,0.204124,0.000000,0.000000


Streaming output truncated to the last 5000 lines.
Kaiden collected 158 cans in his first week, and 259 in his second week for a total of 158 + 259 = 417 cans.
Since his goal is 500, he needs to collect 500 - 417 = 83 more cans. 
Extracted:
Reasoning:
To determine how many more cans Kaiden needs to collect, we first need to calculate the total number of cans he has collected so far, then subtract that from his goal.

Answer:
Kaiden collected 158 cans in his first week, and 259 in his second week for a total of 158 + 259 = 417 cans.
Since his goal is 500, he needs to collect 500 - 417 = 83 more cans.
-------------------- Question:
Calvin has been saving his hair clippings after each haircut to make a wig for his dog. He has gotten 8 haircuts and knows that he needs 2 more to reach his goal. What percentage towards his goal is he? 
Answer:
80 
Response:
<reasoning>
To calculate the percentage towards Calvin's goal, we need to first find out the total number of haircuts he needs. He needs

TrainOutput(global_step=250, training_loss=0.021108752554035163, metrics={'train_runtime': 8032.6075, 'train_samples_per_second': 0.187, 'train_steps_per_second': 0.031, 'total_flos': 0.0, 'train_loss': 0.021108752554035163})

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [7]:
text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "Calculate pi."},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

"Calculating pi (π) is a challenging task, but there are several methods to approximate its value. Here are a few:\n\n**Method 1: Archimedes' Method**\n\nArchimedes' method is an ancient method for approximating pi. He used the Pythagorean theorem to calculate the areas of polygons and then used those areas to estimate pi.\n\n*   For a right-angled triangle with a hypotenuse of length 1 and one leg of length x, the area of the triangle is x/2.\n*   The perimeter of the triangle is 1 + x.\n*   The ratio of the area to the perimeter is x/2 / (1 + x).\n*   By using this ratio, Archimedes was able to calculate pi to be between 3 10/71 and 3 1/7.\n*   He repeated this process with larger and larger polygons, and the result was a more accurate approximation of pi.\n\n**Method 2: Leibniz's Formula**\n\nIn 1671, the German mathematician Gottfried Wilhelm Leibniz developed a formula for calculating pi:\n\nπ = 4 \\* (1 - 1/3 + 1/5 - 1/7 + 1/9 - ...)\n\nThis formula is an infinite series, and the

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [8]:
model.save_lora("grpo_saved_lora")

Now we load the LoRA and test:

In [9]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "Calculate pi."},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

'.reasoning\nTo calculate pi, we will be using the Leibniz formula for pi, which is an infinite series. This formula is based on the idea that the value of pi can be approximated as the sum of an infinite series of terms. The formula is:\n\nπ = 1 - 1/3 + 1/5 - 1/7 + 1/9 - 1/11 + ...\n\nThis formula is an alternating series, where the numerator and denominator of each term alternate between 1 and -1.\n\nAnswer\nUsing the Leibniz formula for pi, we can calculate the value of pi as follows:\n\n1 - 1/3 + 1/5 - 1/7 + 1/9 - 1/11 + 1/13 - 1/15 + ...\n\nUsing a calculator or computer program to sum the first 100 terms of the series, we get:\n\nπ ≈ 3.141592653589793\n\nUsing the first 1000 terms, we get:\n\nπ ≈ 3.14159265358979323846\n\nUsing the first 10,000 terms, we get:\n\nπ ≈ 3.14159265358979323846\n\nThe more terms we use, the closer the approximation becomes to the actual value of pi. However, for most practical purposes, 10 decimal places of precision are sufficient.'

Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if True: model.save_pretrained_merged("llama_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if True: model.push_to_hub_merged("s1lv3rj1nx/llama_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "****")

# Merge to 4bit
if False: model.save_pretrained_merged("llama_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("s1lv3rj1nx/llama_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "****")

# Just LoRA adapters
if False:
    model.save_pretrained("llama_lora")
    tokenizer.save_pretrained("llama_lora")
if False:
    model.push_to_hub("s1lv3rj1nx/llama_lora", token = "****")
    tokenizer.push_to_hub("s1lv3rj1nx/llama_lora", token = "****")

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("llama_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("s1lv3rj1nx/llama_finetune", tokenizer, token = "*****")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("s1lv3rj1nx/llama_finetune", tokenizer, quantization_method = "f16", token = "*****")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("s1lv3rj1nx/llama_finetune", tokenizer, quantization_method = "q4_k_m", token = "*****")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "s1lv3rj1nx/llama_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "*****",
    )

Now, use the `llama_finetune.Q8_0.gguf` file or `llama_finetune.Q4_K_M.gguf` file in llama.cpp.

And we're done!